## Order Items Table - PySpark Cleaning
This table does not need transformations. We validate the data and report any issues found.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, abs as spark_abs
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("OrderItemsCleaning").getOrCreate()

df_order_items = spark.read.csv("../Coursework_dataset/order_items.csv", header=True, inferSchema=True)
df_products = spark.read.csv("../Coursework_dataset/products.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:20:11 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:20:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:20:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:20:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:20:14 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/14 18:20:14 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


### Preview the order_items table

In [2]:
df_order_items.show(5)

+-------------+----------+----------+--------+------------------+----------+
|order_item_id|  order_id|product_id|quantity|unit_price_at_sale|line_total|
+-------------+----------+----------+--------+------------------+----------+
| item_0000001|ord_000001|prod_01446|       4|          16951.04|  67804.16|
| item_0000002|ord_000002|prod_04562|       5|           1449.13|   7245.65|
| item_0000003|ord_000002|prod_04575|       1|            782.99|    782.99|
| item_0000004|ord_000003|prod_03437|       3|            219.07|    657.21|
| item_0000005|ord_000003|prod_03429|       1|           2231.91|   2231.91|
+-------------+----------+----------+--------+------------------+----------+
only showing top 5 rows


### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_order_items.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_order_items.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,order_item_id,919204
1,order_id,919204
2,product_id,919204
3,quantity,919204
4,unit_price_at_sale,919204
5,line_total,919204


### Check which columns have null values

In [4]:
null_counts = df_order_items.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_order_items.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,order_item_id,0
1,order_id,0
2,product_id,0
3,quantity,0
4,unit_price_at_sale,0
5,line_total,0


### Check for duplicate order_item_id rows
Each row should be unique. Duplicates would mean the same item was recorded twice.

In [5]:
duplicates = df_order_items.join(
    df_order_items.groupBy("order_item_id").count().filter(col("count") > 1).select("order_item_id"),
    on="order_item_id",
    how="inner"
)

print("Duplicate order_item_id rows:", duplicates.count())

Duplicate order_item_id rows: 0


### Check 1: Find rows where line_total does not match quantity * unit_price_at_sale
The line_total should always equal quantity multiplied by unit_price_at_sale.
We allow a small rounding tolerance of 0.01.

In [6]:
# Calculate the expected total and compare it to the stored line_total
df_order_items = df_order_items.withColumn(
    "expected_total",
    col("quantity") * col("unit_price_at_sale")
)

# Flag rows where the difference is more than 0.01 (rounding tolerance)
mismatched = df_order_items.filter(
    spark_abs(col("line_total") - col("expected_total")) > 0.01
)

print("Rows where line_total does not match quantity * unit_price_at_sale:", mismatched.count())
mismatched.select("order_item_id", "quantity", "unit_price_at_sale", "line_total", "expected_total").show(10)

Rows where line_total does not match quantity * unit_price_at_sale: 400
+-------------+--------+------------------+----------+--------------+
|order_item_id|quantity|unit_price_at_sale|line_total|expected_total|
+-------------+--------+------------------+----------+--------------+
| item_0005411|       5|           1015.43|   6243.15|       5077.15|
| item_0005562|       1|           8807.25|   9411.14|       8807.25|
| item_0017386|       3|           4289.18|  15769.72|      12867.54|
| item_0020517|       4|          18738.37|  91920.74|      74953.48|
| item_0021545|       2|          29941.34|  68065.65|      59882.68|
| item_0028846|       5|           3465.14|  21385.14|       17325.7|
| item_0029143|       2|            5652.3|  12578.82|       11304.6|
| item_0032063|       4|           1512.66|   7305.88|       6050.64|
| item_0034391|       5|          19344.84| 119471.82|       96724.2|
| item_0035598|       5|           1688.03|   9739.33|       8440.15|
+-------------+---

### Remove the helper column before export

In [7]:
df_order_items = df_order_items.drop("expected_total")

### Check 2: Find orphan product_id records
Some order items may reference a product_id that does not exist in the products table.
These are called orphan records.

In [8]:
# Trim whitespace from product_id in both tables before joining
df_order_items = df_order_items.withColumn("product_id", F.trim(col("product_id")))
df_products = df_products.withColumn("product_id", F.trim(col("product_id")))

# Find order items whose product_id has no matching row in products
orphan_products = df_order_items.join(
    df_products.select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Order items with no matching product:", orphan_products.count())
orphan_products.select("order_item_id", "product_id").show(10)

Order items with no matching product: 250
+-------------+----------+
|order_item_id|product_id|
+-------------+----------+
| item_0002977|prod_05239|
| item_0003912|prod_05184|
| item_0005858|prod_05110|
| item_0011514|prod_05120|
| item_0020570|prod_05135|
| item_0032227|prod_05016|
| item_0032980|prod_05071|
| item_0043336|prod_05124|
| item_0044691|prod_05218|
| item_0044787|prod_05109|
+-------------+----------+
only showing top 10 rows


### Check the schema to confirm data types

In [9]:
df_order_items.printSchema()

root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_at_sale: double (nullable = true)
 |-- line_total: double (nullable = true)



### Export the validated order_items table

In [10]:
df_order_items.coalesce(1).write.csv("../cleaned_dataset/order_items_cleaned.csv", header=True, mode="overwrite")
print("Order items table exported successfully to 'cleaned_dataset/order_items_cleaned.csv'")

Order items table exported successfully to 'cleaned_dataset/order_items_cleaned.csv'
